<a href="https://colab.research.google.com/github/sparshbansal-newton/deep-learning-labs/blob/main/Notebooks/4_pytorch_forward_prop/churn_forward_prop_and_loss_STUDENT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Neural Network with PyTorch — Forward Propagation & Loss (Bank Customer Churn)

In this notebook we build the **first part** of a neural network from scratch with PyTorch, using the **Bank Customer Churn** dataset. Each row is a bank customer; our job is to predict whether the customer **left the bank** (`Churn = 1`) or **stayed** (`Churn = 0`).

We focus on only **two building blocks** today:
1. **Forward propagation** — how the model turns input features into a predicted probability.
2. **Loss** — how we measure how wrong that prediction is.

We will **not** train the model yet (no backpropagation, no gradient updates). The weights stay **random** throughout — the goal is purely to understand *what a forward pass computes* and *what the loss number means*. Training comes next class.

Before any of that, real-world data needs **cleaning**. This dataset forces us to practise three essential steps:
- **Removing unnecessary columns** (IDs and names carry no predictive signal)
- **Label encoding** (a neural network only understands numbers, not text like `France` or `Male`)
- **Standardization** (features live on wildly different scales — Age ~40 vs Balance ~100,000)

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

# Fix the random seed so weight initialization is reproducible across runs
torch.manual_seed(42)

### Load the data

We read the CSV straight from a public URL into a pandas DataFrame and peek at the first few rows.

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/YBI-Foundation/Dataset/main/Bank%20Churn%20Modelling.csv')
df.head()

In [ ]:
df.shape

### Step 1 — Remove unnecessary columns

Look at the columns above. `CustomerId` and `Surname` **identify** a customer but say nothing about whether they will churn — feeding them to the model would only add noise (and let it 'memorise' individuals). We drop them so only genuinely predictive features remain.

> **Hint:** use `df.drop(columns=[...], inplace=True)`.

In [ ]:
# TODO: drop the two identifier columns that carry no predictive signal.
# Hint: df.drop(columns=[...], inplace=True)


In [ ]:
df.head()

### Step 2 — Label encoding (text → numbers)

Two columns are **text**: `Geography` (France / Spain / Germany) and `Gender` (Female / Male). A neural network multiplies inputs by weights, so every feature must be **numeric**. `LabelEncoder` maps each category to an integer (e.g. Female→0, Male→1).

> **Hint:** create a `LabelEncoder()` and call `.fit_transform()` on each text column, assigning the result back to that column.

*(Note: our target `Churn` is already 0/1, so it needs no encoding.)*

In [ ]:
# TODO: convert the 'Geography' and 'Gender' text columns into numbers using LabelEncoder.
# Hint: df['Geography'] = LabelEncoder().fit_transform(df['Geography'])


In [ ]:
df.head()

### Step 3 — Train / test split

We separate the **features** (everything except `Churn`) from the **target** (`Churn`), then hold out 20% of the rows as a test set. `random_state=42` makes the split reproducible.

> **Hint:** features `X = df.drop(columns=['Churn'])`, target `y = df['Churn']`, then `train_test_split(X, y, test_size=0.2, random_state=42)`.

In [ ]:
# TODO: build X (all columns except 'Churn') and y (the 'Churn' column),
# then split into train/test with 20% test size and random_state=42.
# Hint: X = df.drop(columns=['Churn']); y = df['Churn']
# Hint: X_train, X_test, y_train, y_test = train_test_split(...)


### Step 4 — Standardization (scaling)

Features are on very different scales (`Age` ~40, `Balance` ~100000, `Credit Score` ~600). Large-scale features would dominate the weighted sum and slow learning. `StandardScaler` rescales each feature to **mean 0, standard deviation 1**.

> **Important:** `fit_transform` on the **training** set only, then `transform` the test set with the *same* scaler — the test set must never influence the scaling statistics.

In [ ]:
# TODO: create a StandardScaler, fit_transform X_train, and transform X_test.
# Hint: scaler = StandardScaler(); X_train = scaler.fit_transform(X_train); X_test = scaler.transform(X_test)


In [ ]:
X_train

### Step 5 — NumPy arrays → PyTorch tensors

PyTorch operates on **tensors**, so we convert our NumPy arrays. Note that `y_train` / `y_test` are pandas Series, so we take their `.values` first.

> **Hint:** `torch.from_numpy(...)`. For the labels use `y_train.values` and `y_test.values`.

In [ ]:
# TODO: convert X_train, X_test, y_train, y_test into torch tensors.
# Hint: torch.from_numpy(X_train)  ... and for labels torch.from_numpy(y_train.values)


In [ ]:
X_train_tensor.shape

In [ ]:
y_train_tensor.shape

### Step 6 — Defining the model

A single-layer neural network does two things in its `forward` pass:

1. **Linear step** — combine all input features into one number:
$$z = X \cdot \text{weights} + \text{bias}$$
2. **Activation step** — squash that number into a probability between 0 and 1 with the sigmoid function:
$$\hat{y} = \sigma(z) = \frac{1}{1 + e^{-z}}$$

This `forward` pass alone doesn't learn — it just computes a prediction from whatever weights it currently has (right now: random values).

The `loss_function` then compares the prediction $\hat{y}$ to the true label $y$ using **binary cross-entropy**:
$$L = -\big(y \cdot \log(\hat{y}) + (1-y)\cdot \log(1-\hat{y})\big)$$

A high loss means the prediction was far from the truth; a low loss means it was close. We do **not** set `requires_grad` here — no gradients or updates happen this class.

> **Hint:** in `__init__`, `self.weights = torch.rand(X.shape[1], 1, dtype=torch.float64)` and `self.bias = torch.zeros(1, dtype=torch.float64)`. In `forward`, use `torch.matmul` then `torch.sigmoid`.

In [ ]:
class MySimpleNN():

  def __init__(self, X):
    # TODO: initialise self.weights as random values of shape (num_features, 1), dtype=torch.float64
    #       and self.bias as zeros of shape (1,), dtype=torch.float64
    pass

  def forward(self, X):
    # TODO: linear step  z = X @ weights + bias, then activation  y_pred = sigmoid(z); return y_pred
    # Hint: torch.matmul(...) and torch.sigmoid(...)
    pass

  def loss_function(self, y_pred, y):
    # TODO: implement binary cross-entropy.
    # Hint: clamp y_pred to [epsilon, 1-epsilon] with epsilon=1e-7 to avoid log(0),
    #       then loss = -(y*log(y_pred) + (1-y)*log(1-y_pred)).mean()
    pass


### Step 7 — Running a single forward pass

We create the model (random weights), run **one** forward pass on the training data, and compute the loss. No epochs, no weight updates — just to see what a single forward pass + loss looks like.

> **Hint:** `model = MySimpleNN(X_train_tensor)`, then `model.forward(...)`, then `model.loss_function(...)`.

In [ ]:
# TODO: create the model from X_train_tensor, run a single forward pass on X_train_tensor,
#       and compute the loss against y_train_tensor. Then print the first 5 predictions and the loss.
# Hint: model = MySimpleNN(X_train_tensor)


### Evaluation

Let's check accuracy on the test set. The weights were never updated, so they are still random — expect accuracy around chance level. This is the motivation for the next class: we need a way to **update** the weights so the loss goes down and accuracy goes up — that's what backpropagation and gradient descent will do.

In [ ]:
# TODO: with torch.no_grad(), run forward on X_test_tensor, threshold at 0.5 to get labels,
#       compare to y_test_tensor, and print the accuracy.
# Hint: (y_pred_test > 0.5).float()  and  (labels.squeeze() == y_test_tensor).float().mean()


### What's next

In the next class we'll add:
- `requires_grad=True` on the weights and bias
- A training loop that calls `loss.backward()` to compute gradients (backpropagation)
- Gradient-descent updates to actually reduce the loss over multiple epochs

By comparing the loss/accuracy here (random weights) to the loss/accuracy after training, you'll see directly what training accomplishes.